<a href="https://colab.research.google.com/github/data4class/Teaching/blob/main/spectral_clustering_application_(2025).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
def kmeans(P,k):
    n=len(P)
    d=len(P[0])
    # centroids and centroids_old are two different copy of centroids.
    # In centroids_old, we will keep the centroids from the previous iteration:  see the while loop
    clusters = []
    centroids = []
    centroids_old = []
    # Initialize random centroids from the data points
    initial = random.sample(range(n), k)
    for i in initial:
        centroids.append(P[i])
        centroids_old.append(0)
    # While loop is running until centroids gets stable (does not changes)
    while(centroids != centroids_old):
        centroids_old = centroids.copy() # Copy the current centroids in old one so that we can update the current centroids
        centroids = []                   # Empty the current centroids
        clusters = []                    # Initialize the clusters to nil.
        countc = {i: 0 for i in range(k)} # Initialize dictionary to count number of points in each clusters
        for i,x in enumerate(P):    # This loop over each point x in our input
            cdistances = []         # Initialize distance of curent point with all centroids
            for c in centroids_old: # loop over each centroid c
                # computing distance square
                dist = 0
                for j in range(d):
                    dist += (c[j] - x[j])**2
                # append the distance of current point x with current centroid c
                cdistances.append(dist)
            cidx=cdistances.index(min(cdistances)) # find the position of minimum distance, i.e., the cluster number point x belongs to
            countc[cidx] += 1                      # Increase this cluster index, i.e., cluder is count to 1
            clusters.append((i, cidx))             # append cluster id cidx for point x
        # Initialize dictionary sum_cord_centroids to compute total sums for each coordinate of points in each clusters.
        # This is helpful to update centroids later quickly
        sum_cord_centroids = {j: {i: 0 for i in range(d)} for j in range(k)}
        for i,c in clusters:
            for s in range(k):
                if(c == s):
                    for j in range(d):
                        sum_cord_centroids[s][j] += P[i][j]
        # Updating centroids
        for s in sum_cord_centroids:
            for j in range(d):
                sum_cord_centroids[s][j] = sum_cord_centroids[s][j]/countc[s]
            centroids.append(tuple(sum_cord_centroids[s].values()))
    return(clusters,centroids)




In [ ]:
import numpy as np
# If your matrix is too big then use the sparse version of this code from below cell.

def sclusters(W, k):
    n = len(W)
    # Laplacian computation. Note that laplacian is stored in input W, just to save space/memory.
    D_minus_half = 1/np.sqrt(sum(W))
    W = np.multiply(D_minus_half,W).T
    W = -np.multiply(W,D_minus_half)
    np.fill_diagonal(W, 1 + np.diag(W))
    E=np.linalg.eig(W)
    return(kmeans(E[1][:,0:k].tolist(),k))

In [ ]:
### Extra: ignore it for most non-sparse problems.
from scipy.sparse import spdiags, issparse
from scipy.sparse.linalg import lobpcg, LinearOperator

def sclusters_sparse(W, k):
    n = len(W)
    #Laplacian computation. Note that laplacian is stored in input W, just to save space/memory.
    D_minus_half = 1/np.sqrt(sum(W))
    W = np.multiply(D_minus_half,W).T
    W = -np.multiply(W,D_minus_half)
    np.fill_diagonal(W, 1 + np.diag(W))
    X = np.random.rand(n, k)
    eigen = lobpcg(W, X, largest = False)
    return(kmeans(eigen[1].tolist(),k))

In [ ]:
import os
import math
import numpy as np
import time
##
### Test the code above
##
###Example 1
W=np.matrix([[0,0,0,0,0,1,1],
  [0,0,1,1,1,0,0],
  [0,1,0,1,0,0,0],
  [0,1,1,0,1,0,0],
  [0,1,0,1,0,1,0],
  [1,0,0,0,1,0,1],
  [1,0,0,0,0,1,0]])

#sclusters(W,2)
sclusters_sparse(W,2)

/tmp/ipython-input-3-123096211.py:13: UserWarning: The problem size 7 minus the constraints size 0 is too small relative to the block size 2. Using a dense eigensolver instead of LOBPCG iterations.No output of the history of the iterations.
  eigen = lobpcg(W, X, largest = False)


([(0, 0), (1, 1), (2, 1), (3, 1), (4, 1), (5, 0), (6, 0)],
 [(-0.35830498571017605, -0.44175360095456345),
  (-0.389519551181231, 0.30487352158210984)])

In [ ]:
import requests
import pandas as pd
import io

def get_nifty500_symbols():
    # URL of the Nifty 500 list (you may need to update this if the source changes)
    url = "https://archives.nseindia.com/content/indices/ind_nifty100list.csv"

    # Download the CSV file
    response = requests.get(url)

    if response.status_code == 200:
        # Read the CSV content
        df = pd.read_csv(io.StringIO(response.text))

        # Extract the 'Symbol' column
        symbols = df['Symbol'].tolist()

        # Append '.NS' to each symbol for Yahoo Finance
        yahoo_symbols = [f"{symbol}.NS" for symbol in symbols]

        return yahoo_symbols
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None

# Get the list of Yahoo Finance symbols
nifty500_symbols = get_nifty500_symbols()

if nifty500_symbols:
    print(f"Number of stocks: {len(nifty500_symbols)}")
    print("First 10 symbols:")
    print(nifty500_symbols[:10])

Number of stocks: 100
First 10 symbols:
['ABB.NS', 'ADANIENSOL.NS', 'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'AMBUJACEM.NS', 'APOLLOHOSP.NS', 'ASIANPAINT.NS', 'DMART.NS']


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time

# Define stock symbols
stocks = nifty500_symbols #['RELIANCE.NS', 'INFY.NS', 'ADANIPORTS.NS', 'POWERGRID.NS', 'SBIN.NS'] # Add '^CRSLDX' for index NIFTY 500 data for CAPM model

# Download data for each stock individually
all_prices = []
all_returns = []

for stock in stocks[1:50]:
        ticker = yf.Ticker(stock)
        hist = ticker.history(period='5y')

        # Extract close prices
        prices = hist['Close'].dropna()

        # Calculate daily returns
        returns = prices.pct_change().dropna()

        # Store with date index preserved
        all_prices.append(prices)
        all_returns.append(returns)

        print(f"Downloaded {stock}: {len(prices)} days, {len(returns)} returns")

        # Small delay to be respectful to the API
        time.sleep(0.1)

# Create DataFrame for easier analysis
# Create DataFrame with proper date alignment
returns_df = pd.concat([pd.Series(ret, name=stock) for ret, stock in zip(all_returns, stocks)], axis=1)
prices_df = pd.concat([pd.Series(price, name=stock) for price, stock in zip(all_prices, stocks)], axis=1)




Downloaded ADANIENSOL.NS: 476 days, 475 returns
Downloaded ADANIENT.NS: 1240 days, 1239 returns
Downloaded ADANIGREEN.NS: 1240 days, 1239 returns
Downloaded ADANIPORTS.NS: 1240 days, 1239 returns
Downloaded ADANIPOWER.NS: 1240 days, 1239 returns
Downloaded AMBUJACEM.NS: 1240 days, 1239 returns
Downloaded APOLLOHOSP.NS: 1240 days, 1239 returns
Downloaded ASIANPAINT.NS: 1240 days, 1239 returns
Downloaded DMART.NS: 1240 days, 1239 returns
Downloaded AXISBANK.NS: 1240 days, 1239 returns
Downloaded BAJAJ-AUTO.NS: 1240 days, 1239 returns
Downloaded BAJFINANCE.NS: 1240 days, 1239 returns
Downloaded BAJAJFINSV.NS: 1240 days, 1239 returns
Downloaded BAJAJHLDNG.NS: 1240 days, 1239 returns
Downloaded BAJAJHFL.NS: 215 days, 214 returns
Downloaded BANKBARODA.NS: 1240 days, 1239 returns
Downloaded BEL.NS: 1240 days, 1239 returns
Downloaded BPCL.NS: 1240 days, 1239 returns
Downloaded BHARTIARTL.NS: 1240 days, 1239 returns
Downloaded BOSCHLTD.NS: 1240 days, 1239 returns
Downloaded BRITANNIA.NS: 1240 d

In [ ]:
# Drop columns with more than 5 NA values
returns_df = returns_df.dropna(axis=1, thresh=len(returns_df)-5)
prices_df = prices_df.dropna(axis=1, thresh=len(prices_df)-5)

# Drop rows with any NA values
returns_df = returns_df.dropna()
prices_df = prices_df.dropna()

## Compute COrrelation

# Compute correlation matrix of returns
C = returns_df.corr()


#### Create Weight Matrix
f = lambda x: (math.pi - math.acos(x))/math.pi
f2 = np.vectorize(f)

# Apply transformation to correlation matrix
W = f2(C)

# Perform spectral clustering
from sklearn.cluster import SpectralClustering

n_clusters = 10  # adjust as needed
spectral = SpectralClustering(n_clusters=n_clusters, affinity='precomputed', random_state=42)
cluster_labels = spectral.fit_predict(W)

# Add labels to stocks
clustered_stocks = pd.DataFrame({'Stock': C.columns, 'Cluster': cluster_labels})

# Print clustered stocks
print(clustered_stocks)

# Print clusters
for i in range(n_clusters):
    print(f"Cluster {i}: {clustered_stocks[clustered_stocks['Cluster'] == i]['Stock'].tolist()}")

            Stock  Cluster
0   ADANIENSOL.NS        8
1     ADANIENT.NS        8
2   ADANIGREEN.NS        8
3   ADANIPORTS.NS        8
4   ADANIPOWER.NS        8
5    AMBUJACEM.NS        6
6   APOLLOHOSP.NS        3
7   ASIANPAINT.NS        7
8        DMART.NS        5
9     AXISBANK.NS        0
10  BAJAJ-AUTO.NS        1
11  BAJFINANCE.NS        1
12  BAJAJFINSV.NS        7
13    BAJAJHFL.NS        2
14  BANKBARODA.NS        2
15         BEL.NS        9
16        BPCL.NS        5
17  BHARTIARTL.NS        0
18    BOSCHLTD.NS        3
19   BRITANNIA.NS        7
20     CGPOWER.NS        2
21       CANBK.NS        1
22    CHOLAFIN.NS        6
23       CIPLA.NS        9
24   COALINDIA.NS        1
25         DLF.NS        3
26       DABUR.NS        6
27    DIVISLAB.NS        6
28     DRREDDY.NS        0
29     ETERNAL.NS        9
30        GAIL.NS        3
31    GODREJCP.NS        1
32      GRASIM.NS        9
33     HCLTECH.NS        5
34    HDFCBANK.NS        4
35    HDFCLIFE.NS        1
3